In [ ]:
================================MNIST using MLP=========================================
# Import necessary libraries
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt
import numpy as np

# Load the MNIST dataset
(X_train, y_train), (X_test, y_test) = datasets.mnist.load_data()

# Normalize the pixel values
X_train, X_test = X_train / 255.0, X_test / 255.0

# Display the first few images
plt.figure(figsize=(10,10))
for i in range(16):
    plt.subplot(4, 4, i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(X_train[i], cmap=plt.cm.binary)
    plt.xlabel(y_train[i])
plt.show()

# Building the model
model = models.Sequential([
    layers.Flatten(input_shape=(28, 28)), # Flatten the input
    layers.Dense(128, activation='relu'), # First dense layer
    layers.Dropout(0.2),                  # Dropout for regularization
    layers.Dense(10, activation='softmax')# Output layer
])

# Display the model's architecture
model.summary()
# Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    validation_data=(X_test, y_test)
)

#Gradient Descent
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,      # mini-batch gradient descent
    validation_data=(X_test, y_test)
)

# Evaluate the model
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=2)
print(f'\nTest accuracy: {test_acc}')

# Plot training and validation accuracy over epochs
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.legend(loc='lower right')
plt.show()

# Make predictions
predictions = model.predict(X_test)

# Plot some predictions with their true labels
plt.figure(figsize=(10, 10))
for i in range(16):
    plt.subplot(4, 4, i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)

    plt.imshow(X_test[i], cmap=plt.cm.binary)

    predicted_label = np.argmax(predictions[i])
    true_label = y_test[i]

    color = 'blue' if predicted_label == true_label else 'red'
    plt.xlabel(f'{predicted_label} ({true_label})', color=color)

plt.show()


In [ ]:
================================MLP Hyperparameter Tuning=========================================
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Compute confusion matrix
y_pred = np.argmax(predictions, axis=1)
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix
plt.figure(figsize=(10, 10))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=range(10),
    yticklabels=range(10)
)

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Hyperparameter Tuning
neurons_list = [128, 256]
dropout_list = [0.2, 0.3]
batch_sizes = [32, 64]

best_accuracy = 0
best_params = {}

for neurons in neurons_list:
    for dropout in dropout_list:
        for batch in batch_sizes:

            print("\nTraining model with:")
            print("Neurons:", neurons,
                  "Dropout:", dropout,
                  "Batch size:", batch)

            # Build model
            model = models.Sequential([
                layers.Flatten(input_shape=(28,28)),

                layers.Dense(neurons, activation='relu',
                             kernel_regularizer=regularizers.l2(0.001)),
                layers.Dropout(dropout),

                layers.Dense(10, activation='softmax')
            ])

            model.compile(
                optimizer='adam',
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy']
            )

            # Train
            history = model.fit(
                X_train, y_train,
                epochs=5,
                batch_size=batch,
                validation_data=(X_test, y_test),
                verbose=0
            )

            # Evaluate
            loss, acc = model.evaluate(X_test, y_test, verbose=0)

            print("Validation Accuracy:", acc)

            # Save best model
            if acc > best_accuracy:
                best_accuracy = acc
                best_params = {
                    "neurons": neurons,
                    "dropout": dropout,
                    "batch_size": batch
                }

print("\nBest Model Parameters")
print("----------------------")
print("Neurons:", best_params["neurons"])
print("Dropout:", best_params["dropout"])
print("Batch Size:", best_params["batch_size"])
print("Best Accuracy:", best_accuracy)


In [ ]:
====================================MNIST using CNN====================================================

import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt
import numpy as np

# Load MNIST
(X_train, y_train), (X_test, y_test) = datasets.mnist.load_data()

# Normalize
X_train = X_train / 255.0
X_test = X_test / 255.0

# Reshape for CNN (add channel dimension)
X_train = X_train.reshape(-1,28,28,1)
X_test = X_test.reshape(-1,28,28,1)

print("Training shape:", X_train.shape)

model = models.Sequential([

    layers.Conv2D(32,(3,3),activation='relu',input_shape=(28,28,1)),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64,(3,3),activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),

    layers.Dense(128,activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(10,activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test,y_test)
)

test_loss, test_acc = model.evaluate(X_test,y_test)
print("Test Accuracy:", test_acc)


In [ ]:
===============================Face Detection using CNN===================================================

from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    "faces_dataset",
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_data = datagen.flow_from_directory(
    "faces_dataset",
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)
face_model = models.Sequential([

    layers.Conv2D(32,(3,3),activation='relu',input_shape=(128,128,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),

    layers.Dense(128,activation='relu'),
    layers.Dropout(0.4),

    layers.Dense(train_data.num_classes,activation='softmax')
])

face_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = face_model.fit(
    train_data,
    epochs=10,
    validation_data=val_data
)


In [ ]:
=============================================perceptron==========================================

from tensorflow.keras.preprocessing import image
import numpy as np

img = image.load_img("test_face.jpg", target_size=(128,128))
img_array = image.img_to_array(img)/255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = face_model.predict(img_array)

print("Predicted class:", np.argmax(prediction))

model = models.Sequential([
    layers.Flatten(input_shape=(28,28)),  # Convert 2D image to 1D vector
    layers.Dense(10, activation='softmax') # Perceptron output layer
])

model.compile(
    optimizer='sgd',   # Classic perceptron uses gradient descent
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test)
)